In [2]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test'). \
    getOrCreate()

In [ ]:
df_green = spark.read.parquet('data/pq/green/*/*')

In [ ]:
# SELECT
#     date_trunc('hour', lpep_pickup_datetime) AS hour,
#     PULocationID AS zone,
    
#     SUM(total_amount) AS amount,
#     COUNT(1) AS number_records
# FROM
#     green
# WHERE
#     lpep_pickup_datetime >= '2020-01-01'
# GROUP BY
#     1, 2
# ORDER BY
#     1, 2

In [19]:
rdd = df_green.select('lpep_pickup_datetime', 'PULocationID', 'total_amount').rdd

In [7]:
from datetime import datetime

In [8]:
start = datetime(year=2020, month=1, day=1)

def filter_outlires(row):
    return row.lpep_pickup_datetime >= start

In [11]:
rows = rdd.take(10)
row = rows[0]

In [15]:
row

Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 12, 18, 15, 4), PULocationID=41, total_amount=7.88)

In [26]:
def prepare_for_grouping(row):
    hour = row.lpep_pickup_datetime.replace(minute=0, second=0, microsecond=0)
    zone = row.PULocationID
    key = (hour, zone)

    amount = row.total_amount
    count = 1
    value = (amount, count)
    
    return (key, value)

In [29]:
def calculate_revenue(left_value, right_value):
    left_amount, left_count = left_value
    right_amount, right_count = right_value

    output_amount = left_amount + right_amount
    output_count = left_count + right_count
    
    output_value = (output_amount, output_count)

    return (output_amount, output_count)

In [35]:
from collections import namedtuple

In [36]:
RevenueRow = namedtuple('RevenueRow', ['hour', 'zone', 'revenue', 'count'])

In [37]:
def unwrap(row):
    return RevenueRow(hour=row[0][0], zone=row[0][1], revenue=row[1][0], count=row[1][1])

In [44]:
from pyspark.sql import types

result_schema = types.StructType([
    types.StructField('hour', types.TimestampType(), True),
    types.StructField('zone', types.IntegerType(), True),
    types.StructField('revenue', types.DoubleType(), True),
    types.StructField('count', types.IntegerType(), True)
])

In [45]:
df_result = rdd \
    .filter(filter_outlires) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .map(unwrap) \
    .toDF(result_schema)

In [46]:
df_result.schema

StructType([StructField('hour', TimestampType(), True), StructField('zone', IntegerType(), True), StructField('revenue', DoubleType(), True), StructField('count', IntegerType(), True)])